In [ ]:
#!pip install tqdm
#!pip install statsmodels

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from statsmodels.stats.multitest import multipletests

/home/dcm/env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dfm = pd.read_csv("16S_240222Pat_240812Pat/16S_CLR.txt", sep = '\t')
dfm.columns

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="OTU",
    values="Abundance",
    fill_value=min(dfm['Abundance'])
)

In [3]:
#filter df
GroupCol = "Biopsy_collection_date_year"
GroupA = "Year_20"
GroupB = "Year_26"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_ttest(otu):
    # Extract vectors for the OTU
    A_vec = otu_A[otu].values.astype(float)
    B_vec = otu_B[otu].values.astype(float)

    # Drop NaN values to avoid t-test failures
    A_vec = A_vec[~np.isnan(A_vec)]
    B_vec = B_vec[~np.isnan(B_vec)]

    # Require at least 2 values per group
    if len(A_vec) < 2 or len(B_vec) < 2:
        return None

    # Compute descriptive stats
    A_mean = A_vec.mean()
    A_std = A_vec.std(ddof=1)
    B_mean = B_vec.mean()
    B_std = B_vec.std(ddof=1)

    # T-test
    try:
        t_stat, p_value = ttest_ind(A_vec, B_vec, equal_var=False) #Welch’s t-test because equal_var=False
    except Exception:
        return None

    return {
        "otu": otu,
        "T-stat": t_stat,
        "P-value": p_value,
        f"{GroupA}_mean": A_mean,
        f"{GroupA}_std": A_std,
        f"{GroupA}_n": len(A_vec),
        f"{GroupB}_mean": B_mean,
        f"{GroupB}_std": B_std,
        f"{GroupB}_n": len(B_vec)
    }


otus = otu_pa.columns

results_t = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_ttest)(otu)
    for otu in tqdm(otus, desc="Running t-tests", ncols=80)
)


results_t = [r for r in results_t if r is not None]

ttest_df = pd.DataFrame(results_t)

ttest_df["FDR_BH"] = multipletests(ttest_df["P-value"], method="fdr_bh")[1]

#read taxid to merge with genus name
df_taxids = pd.read_csv('16S_240222Pat_240812Pat/16s_OTU_taxids.txt', sep = '\t', usecols = ['#Classification', 'genus'])
df_taxids = df_taxids.drop_duplicates()

ttest_df = pd.merge(ttest_df, df_taxids, left_on = "otu", right_on = "#Classification", how = "left")

ttest_df = ttest_df.drop(["#Classification"], axis=1)

ttest_df.to_csv(f"stats_out/ttest_16S_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running t-tests: 100%|███████████████████████| 389/389 [00:01<00:00, 201.02it/s]


In [4]:
dfm = pd.read_csv("16S_240222Pat_240812Pat/16S_CLR.txt", sep = '\t')
dfm = dfm[dfm["Prog.Nonprog.between.20.26y"] == 'N']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="OTU",
    values="Abundance",
    fill_value=min(dfm['Abundance'])
)

In [5]:
#filter df
GroupCol = "Biopsy_collection_date_year"
GroupA = "Year_20"
GroupB = "Year_26"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_ttest(otu):
    # Extract vectors for the OTU
    A_vec = otu_A[otu].values.astype(float)
    B_vec = otu_B[otu].values.astype(float)

    # Drop NaN values to avoid t-test failures
    A_vec = A_vec[~np.isnan(A_vec)]
    B_vec = B_vec[~np.isnan(B_vec)]

    # Require at least 2 values per group
    if len(A_vec) < 2 or len(B_vec) < 2:
        return None

    # Compute descriptive stats
    A_mean = A_vec.mean()
    A_std = A_vec.std(ddof=1)
    B_mean = B_vec.mean()
    B_std = B_vec.std(ddof=1)

    # T-test
    try:
        t_stat, p_value = ttest_ind(A_vec, B_vec, equal_var=False)
    except Exception:
        return None

    return {
        "otu": otu,
        "T-stat": t_stat,
        "P-value": p_value,
        f"{GroupA}_mean": A_mean,
        f"{GroupA}_std": A_std,
        f"{GroupA}_n": len(A_vec),
        f"{GroupB}_mean": B_mean,
        f"{GroupB}_std": B_std,
        f"{GroupB}_n": len(B_vec)
    }


otus = otu_pa.columns

results_t = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_ttest)(otu)
    for otu in tqdm(otus, desc="Running t-tests", ncols=80)
)

# Remove None
results_t = [r for r in results_t if r is not None]

ttest_df = pd.DataFrame(results_t)

ttest_df["FDR_BH"] = multipletests(ttest_df["P-value"], method="fdr_bh")[1]

#read taxid to merge with genus name
df_taxids = pd.read_csv('16S_240222Pat_240812Pat/16s_OTU_taxids.txt', sep = '\t', usecols = ['#Classification', 'genus'])
df_taxids = df_taxids.drop_duplicates()

ttest_df = pd.merge(ttest_df, df_taxids, left_on = "otu", right_on = "#Classification", how = "left")

ttest_df = ttest_df.drop(["#Classification"], axis=1)

ttest_df.to_csv(f"stats_out/ttest_16S_NonProg_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running t-tests: 100%|██████████████████████| 389/389 [00:00<00:00, 2912.55it/s]


In [6]:
dfm = pd.read_csv("16S_240222Pat_240812Pat/16S_CLR.txt", sep = '\t')
dfm = dfm[dfm["Prog.Nonprog.between.20.26y"] == 'P']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="OTU",
    values="Abundance",
    fill_value=min(dfm['Abundance'])
)

In [7]:
#filter df
GroupCol = "Biopsy_collection_date_year"
GroupA = "Year_20"
GroupB = "Year_26"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_ttest(otu):
    # Extract vectors for the OTU
    A_vec = otu_A[otu].values.astype(float)
    B_vec = otu_B[otu].values.astype(float)

    # Drop NaN values to avoid t-test failures
    A_vec = A_vec[~np.isnan(A_vec)]
    B_vec = B_vec[~np.isnan(B_vec)]

    # Require at least 2 values per group
    if len(A_vec) < 2 or len(B_vec) < 2:
        return None

    # Compute descriptive stats
    A_mean = A_vec.mean()
    A_std = A_vec.std(ddof=1)
    B_mean = B_vec.mean()
    B_std = B_vec.std(ddof=1)

    # T-test
    try:
        t_stat, p_value = ttest_ind(A_vec, B_vec, equal_var=False)
    except Exception:
        return None

    return {
        "otu": otu,
        "T-stat": t_stat,
        "P-value": p_value,
        f"{GroupA}_mean": A_mean,
        f"{GroupA}_std": A_std,
        f"{GroupA}_n": len(A_vec),
        f"{GroupB}_mean": B_mean,
        f"{GroupB}_std": B_std,
        f"{GroupB}_n": len(B_vec)
    }


otus = otu_pa.columns

results_t = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_ttest)(otu)
    for otu in tqdm(otus, desc="Running t-tests", ncols=80)
)

# Remove None
results_t = [r for r in results_t if r is not None]

ttest_df = pd.DataFrame(results_t)

ttest_df["FDR_BH"] = multipletests(ttest_df["P-value"], method="fdr_bh")[1]

#read taxid to merge with genus name
df_taxids = pd.read_csv('16S_240222Pat_240812Pat/16s_OTU_taxids.txt', sep = '\t', usecols = ['#Classification', 'genus'])
df_taxids = df_taxids.drop_duplicates()

ttest_df = pd.merge(ttest_df, df_taxids, left_on = "otu", right_on = "#Classification", how = "left")

ttest_df = ttest_df.drop(["#Classification"], axis=1)

ttest_df.to_csv(f"stats_out/ttest_16S_Prog_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running t-tests: 100%|██████████████████████| 389/389 [00:00<00:00, 2851.31it/s]


In [8]:
dfm = pd.read_csv("16S_240222Pat_240812Pat/16S_CLR.txt", sep = '\t')

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="OTU",
    values="Abundance",
    fill_value=min(dfm['Abundance'])
)

In [9]:
#filter df
GroupCol = "Prog.Nonprog.between.20.26y"
GroupA = "N"
GroupB = "P"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_ttest(otu):
    # Extract vectors for the OTU
    A_vec = otu_A[otu].values.astype(float)
    B_vec = otu_B[otu].values.astype(float)

    # Drop NaN values to avoid t-test failures
    A_vec = A_vec[~np.isnan(A_vec)]
    B_vec = B_vec[~np.isnan(B_vec)]

    # Require at least 2 values per group
    if len(A_vec) < 2 or len(B_vec) < 2:
        return None

    # Compute descriptive stats
    A_mean = A_vec.mean()
    A_std = A_vec.std(ddof=1)
    B_mean = B_vec.mean()
    B_std = B_vec.std(ddof=1)

    # T-test
    try:
        t_stat, p_value = ttest_ind(A_vec, B_vec, equal_var=False)
    except Exception:
        return None

    return {
        "otu": otu,
        "T-stat": t_stat,
        "P-value": p_value,
        f"{GroupA}_mean": A_mean,
        f"{GroupA}_std": A_std,
        f"{GroupA}_n": len(A_vec),
        f"{GroupB}_mean": B_mean,
        f"{GroupB}_std": B_std,
        f"{GroupB}_n": len(B_vec)
    }


otus = otu_pa.columns 

results_t = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_ttest)(otu)
    for otu in tqdm(otus, desc="Running t-tests", ncols=80)
)

# Remove None
results_t = [r for r in results_t if r is not None]

ttest_df = pd.DataFrame(results_t)

ttest_df["FDR_BH"] = multipletests(ttest_df["P-value"], method="fdr_bh")[1]

#read taxid to merge with genus name
df_taxids = pd.read_csv('16S_240222Pat_240812Pat/16s_OTU_taxids.txt', sep = '\t', usecols = ['#Classification', 'genus'])
df_taxids = df_taxids.drop_duplicates()

ttest_df = pd.merge(ttest_df, df_taxids, left_on = "otu", right_on = "#Classification", how = "left")

ttest_df = ttest_df.drop(["#Classification"], axis=1)

ttest_df.to_csv(f"stats_out/ttest_16S_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running t-tests: 100%|██████████████████████| 389/389 [00:00<00:00, 2700.17it/s]


In [10]:
dfm = pd.read_csv("16S_240222Pat_240812Pat/16S_CLR.txt", sep = '\t')
dfm = dfm[dfm["Biopsy_collection_date_year"] == 'Year_20']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="OTU",
    values="Abundance",
    fill_value=min(dfm['Abundance'])
)

In [11]:
#filter df
GroupCol = "Prog.Nonprog.between.20.26y"
GroupA = "N"
GroupB = "P"


metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_ttest(otu):
    # Extract vectors for the OTU
    A_vec = otu_A[otu].values.astype(float)
    B_vec = otu_B[otu].values.astype(float)

    # Drop NaN values to avoid t-test failures
    A_vec = A_vec[~np.isnan(A_vec)]
    B_vec = B_vec[~np.isnan(B_vec)]

    # Require at least 2 values per group
    if len(A_vec) < 2 or len(B_vec) < 2:
        return None

    # Compute descriptive stats
    A_mean = A_vec.mean()
    A_std = A_vec.std(ddof=1)
    B_mean = B_vec.mean()
    B_std = B_vec.std(ddof=1)

    # T-test
    try:
        t_stat, p_value = ttest_ind(A_vec, B_vec, equal_var=False)
    except Exception:
        return None

    return {
        "otu": otu,
        "T-stat": t_stat,
        "P-value": p_value,
        f"{GroupA}_mean": A_mean,
        f"{GroupA}_std": A_std,
        f"{GroupA}_n": len(A_vec),
        f"{GroupB}_mean": B_mean,
        f"{GroupB}_std": B_std,
        f"{GroupB}_n": len(B_vec)
    }

otus = otu_pa.columns

results_t = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_ttest)(otu)
    for otu in tqdm(otus, desc="Running t-tests", ncols=80)
)

results_t = [r for r in results_t if r is not None]

ttest_df = pd.DataFrame(results_t)

ttest_df["FDR_BH"] = multipletests(ttest_df["P-value"], method="fdr_bh")[1]

#read taxid to merge with genus name
df_taxids = pd.read_csv('16S_240222Pat_240812Pat/16s_OTU_taxids.txt', sep = '\t', usecols = ['#Classification', 'genus'])
df_taxids = df_taxids.drop_duplicates()

ttest_df = pd.merge(ttest_df, df_taxids, left_on = "otu", right_on = "#Classification", how = "left")

ttest_df = ttest_df.drop(["#Classification"], axis=1)

ttest_df.to_csv(f"stats_out/ttest_16S_Year_20_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running t-tests: 100%|██████████████████████| 389/389 [00:00<00:00, 2917.39it/s]


In [12]:
dfm = pd.read_csv("16S_240222Pat_240812Pat/16S_CLR.txt", sep = '\t')
dfm = dfm[dfm["Biopsy_collection_date_year"] == 'Year_26']

otu_pa = dfm.pivot_table(
    index="MIT_Accession",
    columns="OTU",
    values="Abundance",
    fill_value=min(dfm['Abundance'])
)

In [13]:
#filter df
GroupCol = "Prog.Nonprog.between.20.26y"
GroupA = "N"
GroupB = "P"

metadata = dfm[["MIT_Accession", f"{GroupCol}"]].drop_duplicates()
metadata = metadata.set_index("MIT_Accession")

A_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupA}"].index
B_ids = metadata[metadata[f"{GroupCol}"] == f"{GroupB}"].index

otu_A = otu_pa.loc[A_ids]
otu_B = otu_pa.loc[B_ids]

def run_ttest(otu):
    # Extract vectors for the OTU
    A_vec = otu_A[otu].values.astype(float)
    B_vec = otu_B[otu].values.astype(float)

    # Drop NaN values to avoid t-test failures
    A_vec = A_vec[~np.isnan(A_vec)]
    B_vec = B_vec[~np.isnan(B_vec)]

    # Require at least 2 values per group
    if len(A_vec) < 2 or len(B_vec) < 2:
        return None

    # Compute descriptive stats
    A_mean = A_vec.mean()
    A_std = A_vec.std(ddof=1)
    B_mean = B_vec.mean()
    B_std = B_vec.std(ddof=1)

    # T-test
    try:
        t_stat, p_value = ttest_ind(A_vec, B_vec, equal_var=False)
    except Exception:
        return None

    return {
        "otu": otu,
        "T-stat": t_stat,
        "P-value": p_value,
        f"{GroupA}_mean": A_mean,
        f"{GroupA}_std": A_std,
        f"{GroupA}_n": len(A_vec),
        f"{GroupB}_mean": B_mean,
        f"{GroupB}_std": B_std,
        f"{GroupB}_n": len(B_vec)
    }

otus = otu_pa.columns

results_t = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_ttest)(otu)
    for otu in tqdm(otus, desc="Running t-tests", ncols=80)
)

results_t = [r for r in results_t if r is not None]

ttest_df = pd.DataFrame(results_t)

ttest_df["FDR_BH"] = multipletests(ttest_df["P-value"], method="fdr_bh")[1]

#read taxid to merge with genus name
df_taxids = pd.read_csv('16S_240222Pat_240812Pat/16s_OTU_taxids.txt', sep = '\t', usecols = ['#Classification', 'genus'])
df_taxids = df_taxids.drop_duplicates()

ttest_df = pd.merge(ttest_df, df_taxids, left_on = "otu", right_on = "#Classification", how = "left")

ttest_df = ttest_df.drop(["#Classification"], axis=1)

ttest_df.to_csv(f"stats_out/ttest_16S_Year_26_{GroupA}_vs_{GroupB}_results.txt", sep="\t", index=False)

Running t-tests: 100%|██████████████████████| 389/389 [00:00<00:00, 2811.96it/s]
